# Naturalezza: secondo giudice su 61 testi completi

Questo notebook confronta **Llama 3.3 70B** (`llama-3.3-70b-versatile`) con il giudice Qwen già eseguito. Usa soltanto i **61 originali con tutte le 12 risposte Qwen valide**: 17 Essay, 20 Reuters, 24 WritingPrompts. I due pannelli parziali del checkpoint del 18 novembre sono esclusi; le loro risposte restano conservate nell'archivio originale.

Per ciascun originale si presentano le stesse quattro riscritture, con gli stessi messaggi, mapping T1–T4, quattro ordini bilanciati e tre ripetizioni. Sono **732 chiamate previste** al nuovo giudice. I pesi dei modelli non vengono scaricati: basta **CPU, Internet e il Secret `GROQ_API_KEY`** abilitato nelle impostazioni Kaggle.

Il riferimento Qwen è già incluso nella repository: per il primo avvio non occorre aggiungere il vecchio ZIP agli Input. La disponibilità Llama viene verificata nel catalogo autenticato; se manca, l'esecuzione si ferma senza sostituirlo con un altro modello. Nessuna valutazione LLM viene considerata annotazione umana.


## 1. Configurazione

Per avviare le chiamate modificare solo `RUN_API = True`. Il nuovo run ha una cartella distinta da quella Qwen. Il limite di 900 POST comprende tutti i tentativi, anche i retry e quelli delle riprese; le chiamate pianificate sono 732.

Alla prima esecuzione lasciare `RESUME_ZIP = ""`. In una sessione successiva caricare lo ZIP **del nuovo run Llama** negli Input e usare `RESUME_ZIP = "auto"`; funzionano anche un percorso ZIP o una cartella già estratta. Non usare il checkpoint Qwen come ripresa Llama.


In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, os, shutil, stat, subprocess, sys, tempfile, time, zipfile

RUN_API = False
RESUME_ZIP = ""                  # Prima volta vuoto; poi "auto" per il checkpoint Llama negli Input
RUN_SUFFIX = "v1"
JUDGE_MODEL = "llama-3.3-70b-versatile"
REPO_REF = "main"
MAX_POST_ATTEMPTS = 900
MIN_REQUEST_INTERVAL = 8.0
BOOTSTRAP = 2000
SEED = 42
REFERENCE_SHA256 = "c834bf2d03eb3735c93fe105d7a937983f0c44a468784b1969ec997d46f6dee4"

assert JUDGE_MODEL == "llama-3.3-70b-versatile", "Questo confronto fissa esplicitamente Llama 3.3 70B."
assert Path("/kaggle/working").is_dir()
RUN = Path("/kaggle/working") / f"humanization_naturalness_panel_llama70b_{RUN_SUFFIX}"
REPO = Path("/kaggle/working/.humanization_naturalness_comparison_code/Humanization")
RUN.mkdir(parents=True, exist_ok=True)
MPL_CACHE = Path("/kaggle/working/.humanization_naturalness_comparison_cache/matplotlib")
MPL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(MPL_CACHE)
frozen_settings = {"mode": "cross_family_complete_panels", "model": JUDGE_MODEL,
                   "reference_checkpoint_sha256": REFERENCE_SHA256, "seed": SEED}
print("Giudice:", JUDGE_MODEL, "| API attive:", RUN_API)
print("Cartella nuova:", RUN)


## 2. Riprendere soltanto il nuovo giudice

La prima volta non ci sono risposte Llama da recuperare. Per una ripresa si controllano modello, riferimento Qwen e codice congelato; le risposte Qwen non vengono inserite nel registro Llama. Gli Input sono letti senza modificarli.


In [ ]:
def matching_checkpoint_directories(root, settings):
    matches = []
    for path in root.rglob("runtime_info.json"):
        folder = path.parent
        if not all((folder / name).is_file() for name in
                   ["judge_plan.json", "judgments.jsonl", "api_attempts.jsonl"]):
            continue
        try:
            metadata = json.loads(path.read_text())
        except (OSError, ValueError):
            continue
        if metadata.get("settings") == settings:
            matches.append(folder)
    return sorted(set(matches))

resume_source = RESUME_ZIP
if resume_source == "auto":
    local_files = [RUN / name for name in
                   ["runtime_info.json", "judge_plan.json", "judgments.jsonl", "api_attempts.jsonl"]]
    if all(path.is_file() for path in local_files) and (RUN / "judgments.jsonl").stat().st_size:
        resume_source = ""
        print("Riprendo i file già presenti nella cartella del run.")
    else:
        matches = matching_checkpoint_directories(Path("/kaggle/input"), frozen_settings)
        assert len(matches) == 1, (
            "Serve un unico checkpoint estratto compatibile con modello e riferimento. "
            f"Cartelle trovate: {[str(path) for path in matches]}. "
            "Indicare esplicitamente il percorso corretto in RESUME_ZIP se necessario."
        )
        resume_source = str(matches[0])

if resume_source:
    resume_path = Path(resume_source)
    assert resume_path.exists(), f"Checkpoint da riprendere assente: {resume_path}"
    if resume_path.is_dir():
        if not (resume_path / "runtime_info.json").is_file():
            matches = matching_checkpoint_directories(resume_path, frozen_settings)
            assert len(matches) == 1, "Indicare la cartella che contiene i file del checkpoint."
            resume_path = matches[0]
        metadata = json.loads((resume_path / "runtime_info.json").read_text())
        assert metadata["settings"] == frozen_settings, "Parametri diversi dal checkpoint: controllare modello e riferimento."
        assert metadata["requested_ref"] == REPO_REF, "REPO_REF diverso dal checkpoint."
        assert all((resume_path / name).is_file() for name in
                   ["judge_plan.json", "judgments.jsonl", "api_attempts.jsonl"]), "Checkpoint incompleto."
        files = [path for path in resume_path.rglob("*") if path.is_file()
                 and path.suffix.lower() in {".json", ".jsonl", ".csv", ".png"}]
        assert sum(path.stat().st_size for path in files) <= 512 * 2**20, "Checkpoint troppo grande."
        resume_cache = RUN.parent / ".humanization_naturalness_comparison_cache"
        resume_cache.mkdir(parents=True, exist_ok=True)
        repacked = resume_cache / "checkpoint_resume.zip"
        with zipfile.ZipFile(repacked, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            for path in sorted(files):
                archive.write(path, path.relative_to(resume_path).as_posix())
        print("Checkpoint letto dalla cartella Input:", resume_path)
        resume_path = repacked
    assert resume_path.is_file(), f"Checkpoint ZIP non trovato: {resume_path}"
    with zipfile.ZipFile(resume_path) as archive:
        assert "runtime_info.json" in archive.namelist(), "Lo ZIP non contiene runtime_info.json alla radice."
        metadata = json.loads(archive.read("runtime_info.json"))
        assert metadata["settings"] == frozen_settings, "Parametri diversi dal checkpoint: controllare modello e riferimento."
        assert metadata["requested_ref"] == REPO_REF, "REPO_REF diverso dal checkpoint."
        assert sum(item.file_size for item in archive.infolist()) <= 512 * 2**20, "ZIP di risultati troppo grande."
        for item in archive.infolist():
            assert not any(part in {".", ".."} for part in item.filename.split("/")), "Componenti ZIP non ammesse."
            relative = Path(item.filename)
            assert not relative.is_absolute() and ".." not in relative.parts and "\\" not in item.filename, "Percorso ZIP non valido."
            assert relative.parts and relative.parts[0] != ".", "Percorso ZIP vuoto."
            assert not stat.S_ISLNK(item.external_attr >> 16), "I collegamenti simbolici non sono ammessi."
            target = (RUN / relative).resolve()
            assert target == RUN.resolve() or RUN.resolve() in target.parents, "Percorso esterno alla cartella del run."
            if item.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            assert target.suffix.lower() in {".json", ".jsonl", ".csv", ".png"}, f"File inatteso nello ZIP: {relative}"
            content = archive.read(item)
            if target.exists():
                assert target.read_bytes() == content, f"File diverso già presente: {relative}. Usare una nuova cartella."
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(content)
    print("Risultati precedenti ripristinati senza sovrascritture.")

runtime_path = RUN / "runtime_info.json"
saved_runtime = json.loads(runtime_path.read_text()) if runtime_path.exists() else None
if saved_runtime:
    assert saved_runtime["settings"] == frozen_settings, "Parametri diversi dal run precedente: scegliere un nuovo RUN_SUFFIX."
    assert saved_runtime["requested_ref"] == REPO_REF, "REPO_REF diverso: ripristinare il riferimento precedente o cambiare cartella."
    print("Commit congelato per la ripresa:", saved_runtime["code_commit"])
    for name, label in [("judgments.jsonl", "Risposte già salvate"), ("api_attempts.jsonl", "Tentativi già registrati")]:
        path = RUN / name
        count = sum(bool(line.strip()) for line in path.read_text().splitlines()) if path.exists() else 0
        print(label + ":", count)
else:
    print("Nuovo run: nessuna risposta precedente da recuperare.")


## 3. Codice e librerie

Il notebook usa una copia distinta della repository **Humanization**. Installa soltanto eventuali librerie mancanti per richieste, tabelle e grafici. Nelle riprese riutilizza il commit della prima esecuzione Llama.


In [ ]:
repository_url = "https://github.com/FrancescoM00/Humanization.git"
REPO.parent.mkdir(parents=True, exist_ok=True)
fresh_clone = not REPO.exists()
if fresh_clone:
    subprocess.run(["git", "clone", "--depth", "1", repository_url, str(REPO)], check=True)
else:
    assert (REPO / ".git").is_dir(), f"Cartella esistente non riconosciuta come repository: {REPO}"
    dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True)
    assert not dirty.strip(), "La copia del codice ha modifiche locali: conservarle e scegliere un'altra cartella REPO."
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO, text=True).strip()
    assert remote.removesuffix(".git") == repository_url.removesuffix(".git"), "Repository diversa da quella prevista."

selected_ref = saved_runtime["code_commit"] if saved_runtime else REPO_REF
if not (fresh_clone and selected_ref == "main"):
    subprocess.run(["git", "fetch", "--depth", "1", "origin", selected_ref], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
CODE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert (REPO / "scripts/compare_naturalness_judges.py").is_file(), "Il riferimento scelto non contiene il nuovo script."
print("Repository:", REPO)
print("Commit utilizzato:", CODE_COMMIT)

minimal_packages = {
    "requests": "requests>=2.32,<3", "numpy": "numpy>=1.26,<3",
    "pandas": "pandas>=2.2,<4", "matplotlib": "matplotlib>=3.8,<4",
}
missing = [requirement for module, requirement in minimal_packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
import pandas as pd
from IPython.display import FileLink, Image, display
from importlib.metadata import version
LIBRARY_VERSIONS = {name: version(name) for name in minimal_packages}
print("Librerie:", LIBRARY_VERSIONS)


## 4. Riferimento Qwen congelato e comandi

L'archivio incluso è quello caricato il 18 novembre: 747 risposte valide, una scartata, **61 pannelli completi e due parziali**. Il filtro conserva esclusivamente i 61 completi, senza scegliere in base ai voti e senza tagliare retroattivamente a 60 o riequilibrare i domini.

`reference/` contiene piano e risultati Qwen dei completi; la radice contiene soltanto piano, risposte e tentativi del nuovo giudice. Il manifest `comparison_setup.json` documenta il campione e le esclusioni. Per Llama si omette l'opzione API `reasoning_effort`: è un modello non reasoning. Nessun parametro viene cambiato automaticamente dopo un errore.


In [ ]:
REFERENCE = REPO / "exports/naturalness_judge_comparison/qwen_reference_20261118.zip"
assert REFERENCE.is_file(), "Il commit scelto non contiene l'archivio di riferimento."
assert hashlib.sha256(REFERENCE.read_bytes()).hexdigest() == REFERENCE_SHA256, "Riferimento diverso da quello congelato."
runtime = {"settings": frozen_settings, "requested_ref": REPO_REF, "code_commit": CODE_COMMIT,
           "python": sys.version.split()[0], "library_versions": LIBRARY_VERSIONS}
if saved_runtime:
    assert saved_runtime["code_commit"] == CODE_COMMIT
else:
    runtime_path.write_text(json.dumps(runtime, ensure_ascii=False, indent=2) + "\n")

SCRIPT = REPO / "scripts/compare_naturalness_judges.py"
def comparison_command(execute=False):
    command = [sys.executable, str(SCRIPT), "--reference", str(REFERENCE), "--output", str(RUN),
               "--model", JUDGE_MODEL, "--max-requests", str(MAX_POST_ATTEMPTS),
               "--min-request-interval", str(MIN_REQUEST_INTERVAL), "--bootstrap", str(BOOTSTRAP),
               "--seed", str(SEED)]
    if execute:
        command.append("--execute")
    return command

def load_json(name):
    path = RUN / name
    return json.loads(path.read_text()) if path.is_file() else {}

def read_rows(name):
    path = RUN / name
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()] if path.is_file() else []


## 5. Preparare e controllare senza chiamate API

Devono risultare **61 testi** e **732 chiamate pianificate**. Il contatore Llama parte da zero; i 767 tentativi del primo giudice non consumano il budget del secondo. Questa preparazione non legge il Secret.


In [ ]:
offline_env = dict(os.environ)
offline_env.pop("GROQ_API_KEY", None)
subprocess.run(comparison_command(False), cwd=REPO, env=offline_env, check=True)
setup = load_json("comparison_setup.json")
plan = load_json("judge_plan.json")
assert setup["n_complete_reference_sources"] == 61
assert plan["expected_completion_requests"] == 732
print("Testi completi Qwen inclusi:", setup["n_complete_reference_sources"])
print("Pannelli Qwen parziali esclusi:", setup["n_excluded_partial_sources"])
print("Domini:", setup["complete_sources_by_domain"])
print("Chiamate Llama previste:", plan["expected_completion_requests"])
print("Risposte Llama già salvate:", len(read_rows("judgments.jsonl")))


## 6. Chiamate al nuovo giudice

Con `RUN_API = True` si verifica il modello nel catalogo e si inviano le richieste mancanti. La stessa cella riprende dai giudizi già registrati. Una quota esaurita lascia il run parziale: conservare comunque lo ZIP e riprendere più avanti. I limiti del nuovo modello possono differire da quelli Qwen.

Le risposte con formato non valido restano scartate e non sono ripetute fino a ottenere un voto accettabile. Un pannello Llama incompleto è escluso da **entrambi** i lati del confronto finale.


In [ ]:
planned_ids = {item["request_id"] for item in plan["requests"]}
answered_ids = {row["request_id"] for row in read_rows("judgments.jsonl")}
if RUN_API and planned_ids - answered_ids:
    from kaggle_secrets import UserSecretsClient
    api_secret = UserSecretsClient().get_secret("GROQ_API_KEY")
    assert isinstance(api_secret, str) and api_secret.strip(), "Secret GROQ_API_KEY non disponibile."
    api_env = dict(os.environ)
    api_env["GROQ_API_KEY"] = api_secret.strip()
    started = time.monotonic()
    try:
        outcome = subprocess.run(comparison_command(True), cwd=REPO, env=api_env, check=False)
    finally:
        api_env.pop("GROQ_API_KEY", None)
        del api_secret
    print("Durata di questa esecuzione, minuti:", round((time.monotonic()-started)/60, 2))
    if outcome.returncode:
        print("Il comando si è fermato: controllare l'errore e conservare comunque lo ZIP.")
elif not RUN_API:
    print("RUN_API=False: nessuna chiamata e nessuna chiave letta.")
else:
    print("Tutte le risposte sono già registrate; nessuna nuova chiamata.")
manifest = load_json("judge_run.json")
for name in ["status", "blocked_reason", "completed_valid_requests", "expected_completion_requests",
             "n_complete_source_panels", "generation_post_attempts", "token_usage"]:
    print(name + ":", manifest.get(name))


## 7. Confronto sui testi completi per entrambi

La cartella `comparison/` confronta esclusivamente gli originali con tutte le dodici risposte valide per **entrambi** i giudici. I dati parziali restano conservati ma non entrano nelle medie appaiate. All'inizio il confronto è vuoto: non si inventano voti Llama.

Le tabelle documentano medie per metodo, cambiamenti fra giudici, contrasti fra riscrittori, concordanza dei voti e sensibilità alla posizione. Gli intervalli usano gli originali/gruppi di fonte, non le chiamate ripetute come osservazioni indipendenti. Sono confronti esplorativi: il secondo giudice non costituisce validazione umana.


In [ ]:
subprocess.run(comparison_command(False), cwd=REPO, env=offline_env, check=True)
summary = load_json("comparison/summary.json")
print("Originali di riferimento:", summary.get("n_complete_reference"))
print("Originali completi per entrambi:", summary.get("n_common_complete_sources"))
print("Stato confronto:", summary.get("status"))
comparison_folder = RUN / "comparison"
for name in ["system_summary_by_judge.csv", "humanizer_contrast_shifts.csv",
             "system_shifts.csv", "ordinal_agreement.csv", "position_effects_by_judge.csv"]:
    path = comparison_folder / name
    if path.is_file():
        try:
            frame = pd.read_csv(path)
        except pd.errors.EmptyDataError:
            continue
        print(name, "— righe:", len(frame))
        display(frame.head(10))
for figure in sorted(comparison_folder.glob("*.png")):
    display(Image(filename=str(figure)))
if not summary.get("n_common_complete_sources"):
    print("Il confronto sarà disponibile dopo le prime dodici risposte Llama valide per un originale.")


## 8. Scaricare e riprendere

Lo ZIP include il riferimento Qwen filtrato, il nuovo piano, i giudizi Llama disponibili, tabelle e grafici. Conservare sempre l'archivio più recente, anche se il run è parziale. Nella sessione successiva aggiungerlo agli Input, impostare `RESUME_ZIP = "auto"` e rieseguire questo notebook. Non serve riavviare gli humanizer, i detector o i modelli semantici.


In [ ]:
archive_path = Path(shutil.make_archive(str(RUN), "zip", root_dir=RUN))
with zipfile.ZipFile(archive_path) as archive:
    assert not any(Path(name).name == ".env" for name in archive.namelist())
    assert not any(name.startswith((".git/", ".venv/")) for name in archive.namelist())
print("Archivio da scaricare:", archive_path)
print("Dimensione MiB:", round(archive_path.stat().st_size / 2**20, 2))
display(FileLink(str(archive_path)))
